Step 1: Install Required Dependencies

In [ ]:
! pip install --upgrade langchain openai langchain-openai chromadb  -U langchain-community tiktoken pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.9 MB/s eta 0:00:00

Step 2: Import Necessary Libraries and Configure the LLM

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain import LLMChain, PromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.docstore.document import Document
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from google.colab import userdata
from langchain.llms import OpenAI
from pathlib import Path
import os
import json
import fitz

# Retrieve the OpenAI API key
openai_api_key = userdata.get("OPENAIKEY")

# Configure the OpenAI model
llmModel = ChatOpenAI(
    temperature=0,
    verbose=True,
    openai_api_key=openai_api_key,
    model="gpt-3.5-turbo"
)


Step 3: Define paths

In [ ]:
# Paths
submissions_folder = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Assignments"
rubric_path = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Rubrics/assessment_rubric.txt"
reviewer_prompt_path = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Rubrics/reviewer prompt 5.txt"
question_path = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Question.pdf"
output_dir = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Outputs"
output_file = os.path.join(output_dir, "results.json")

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)


Step 4: Load rubric, reviewer prompt, and assignment question

In [ ]:
# Load rubric and reviewer prompt
with open(rubric_path, "r") as file:
    rubric_and_prompt_text = file.read()

with open(reviewer_prompt_path, "r") as file:
    reviewer_prompt_text = file.read()

# Function to extract text from PDFs
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with fitz.open(pdf_path) as pdf:
            for page in pdf:
                text += page.get_text() + "\n"
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
    return text.strip()

# Extract assignment question
assignment_question = extract_text_from_pdf(question_path)


Step 5: Load domain knowledge documents

In [ ]:
domain_knowledge_folder = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Knowledge Base"
domain_pdfs = [os.path.join(domain_knowledge_folder, f) for f in os.listdir(domain_knowledge_folder) if f.endswith('.pdf')]

# Function to extract text from PDFs for knowledge base
def extract_text_for_knowledge_base(pdf_path):
    text = ""
    try:
        with fitz.open(pdf_path) as pdf:
            for page in pdf:
                text += page.get_text() + "\n"
    except Exception as e:
        print(f"Error processing knowledge base PDF {pdf_path}: {e}")
    return text.strip()

# Prepare documents for the vector store
knowledge_base_docs = []
for pdf in domain_pdfs:
    content = extract_text_for_knowledge_base(pdf)
    if content:  # Only add non-empty documents
        knowledge_base_docs.append(Document(page_content=content, metadata={"source": pdf}))


Step 6: Split text into chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_documents = []
for doc in knowledge_base_docs:
    chunks = text_splitter.split_text(doc.page_content)
    for chunk in chunks:
        split_documents.append(Document(page_content=chunk, metadata=doc.metadata))

Step 7: Initialize vector store

In [ ]:
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
vectorstore = Chroma.from_documents(split_documents, embeddings)
retriever = vectorstore.as_retriever()

Step 8: Create the grading chain

In [ ]:
grading_prompt = PromptTemplate(
    input_variables=["assignment_question", "assignment_content", "rubric", "retrieved_context"],
    template=rubric_and_prompt_text
)

grading_chain = LLMChain(
    llm=llmModel,
    prompt=grading_prompt
)


<ipython-input-13-f5469d929d68>:6: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  grading_chain = LLMChain(


Step 8: Grading Agent with RetrievalQA

In [ ]:
grading_agent = RetrievalQA.from_chain_type(
    llm=llmModel,
    retriever=retriever,
    chain_type="stuff",
    verbose=True
)

Step 9: Define the reviewer chain

In [ ]:
# Step: Define the Reviewer Chain
reviewer_prompt = PromptTemplate(
    input_variables=["feedback", "rubric"],
    template=reviewer_prompt_text
)

reviewer_chain = LLMChain(
    llm=ChatOpenAI(model_name="gpt-4", temperature=0, openai_api_key=openai_api_key),
    prompt=reviewer_prompt
)


Step 9: Process all submissions

In [ ]:
results = []  # List to store results for all submissions

for file_name in os.listdir(submissions_folder):
    if file_name.endswith(".pdf"):
        try:
            submission_path = os.path.join(submissions_folder, file_name)
            print(f"Processing file: {file_name}")

            # Extract assignment content
            assignment_content = extract_text_from_pdf(submission_path)
            if not assignment_content:
                print(f"Skipping empty or unreadable file: {file_name}")
                continue

            # Retrieve context from the knowledge base
            retrieved_context = grading_agent({"query": assignment_content})["result"]

            # Run grading chain
            grading_feedback = grading_chain.run({
                "assignment_question": assignment_question,
                "assignment_content": assignment_content,
                "rubric": rubric_and_prompt_text,
                "retrieved_context": retrieved_context
            })
            print(f"Grading Feedback for {file_name}:\n{grading_feedback}")

            # Run reviewer chain
            reviewer_feedback = reviewer_chain.run({
                "feedback": grading_feedback,
                "rubric": rubric_and_prompt_text
            })
            print(f"Reviewer Feedback for {file_name}:\n{reviewer_feedback}")

            # Append results
            results.append({
                "file_name": file_name,
                "grading_feedback": grading_feedback,
                "reviewer_feedback": reviewer_feedback
            })

        except Exception as e:
            print(f"Error processing file {file_name}: {e}")

# Save results to a JSON file
with open(output_file, "w") as file:
    json.dump(results, file, indent=4)

print(f"Results saved to {output_file}")


Processing file: Submission 8 -TIMG-5103-Assignment1.pdf


> Entering new RetrievalQA chain...

> Finished chain.
Grading Feedback for Submission 8 -TIMG-5103-Assignment1.pdf:
- Selected Solution: 
   - Score: 4
   - Feedback: The solution uses an LLM-based component, but its role within the business context lacks clarity. It would be beneficial to further explain how this component is essential to achieving the assignment's objectives.

- Disruption Analysis:
   - Score: 3
   - Feedback: A disruption analysis is provided, but it does not fully align with disruption theory. Consider providing more specific examples of how the selected solution could disrupt existing market leaders.

- Integration Analysis:
   - Score: 2
   - Feedback: Only one integration challenge criterion is analyzed. It would be helpful to identify and analyze more criteria related to integration challenges to provide a comprehensive analysis.

- Misuse Risks Analysis:
   - Score: 4
   - Feedback: Two to four misus